In [2]:
import pandas as pd
true_df=pd.read_csv(r'C:\Users\RISHIKA REDDY\OneDrive\Documents\mini-project\data\True.csv')
fake_df=pd.read_csv(r'C:\Users\RISHIKA REDDY\OneDrive\Documents\mini-project\data\Fake.csv')

In [3]:
# Assign Labels: 0 for real (True) ,1 for Fake
true_df['label']=0
fake_df['label']=1

# Merge them into one dataframe
df=pd.concat([true_df,fake_df],axis=0).reset_index(drop=True)


In [4]:
# shuffle the data( model doesn't see all real then all fake)
df=df.sample(frac=1).reset_index(drop=True)

# combine title and text for a richer input
df['full_text']=df['title']+" "+df['text']

print(f"Dataset Merged. Total samples : {len(df)}")
print(df[['full_text', 'label']].head())

Dataset Merged. Total samples : 44898
                                           full_text  label
0  Trump mulls national security adviser pick, Wh...      0
1  Boiler Room EP #115 – Very Fake News & The Sla...      1
2  Judge criticized by Trump unseals documents in...      0
3  WOW WIKILEAKS! DNC Planned Big Federal Rewards...      1
4  Trump not planning to invoke executive privile...      0


In [5]:
import re
import torch
from transformers import DistilBertTokenizer

# 1.initialize the tokenizer 
tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def full_nlp_preprocessing(text):
     #1.reomove ISOT-specific headers like Reuters
     text=re.sub(r'^.*?\(Reuters\)\s*-', '',text)
     # 2.remove URLs HTML tags and Newline
     text=re.sub(r'https?://\S+|www\.\S+', '',text)
     # 3.Basic normalization
     text=text.lower().strip()
     #4.Tokenization ,Padding , and Truncation
     encoded_input=tokenizer(
          text,
          padding='max_length',
          truncation=True,
          max_length=512,
          return_tensors='pt'
     )
     return encoded_input

C:\Users\RISHIKA REDDY\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df_small=df.sample(frac=0.1)

In [7]:
sample_news = "WASHINGTON (Reuters) - The White House announced new tech policies today at https://whitehouse.gov"
processed_data = full_nlp_preprocessing(sample_news)

print("Preprocessed Input IDs Shape:", processed_data['input_ids'].shape)
print("First 10 Tokens:", tokenizer.convert_ids_to_tokens(processed_data['input_ids'][0][:10]))

Preprocessed Input IDs Shape: torch.Size([1, 512])
First 10 Tokens: ['[CLS]', 'the', 'white', 'house', 'announced', 'new', 'tech', 'policies', 'today', 'at']


In [10]:
from pandas.io.pickle import to_pickle
# save to pickle_file(fast and preserves data types)
df.to_pickle(r'C:\Users\RISHIKA REDDY\OneDrive\Documents\mini-project\data\merged_isot_data.pkl')